In [ ]:
import sys
sys.path.insert(0, '/app')

from datetime import datetime
from connectors.clickhouse_client import ClickHouseClient

In [ ]:
ref_date = datetime.now().strftime("%Y-%m-%d")
ref_date


In [ ]:
client = ClickHouseClient()


In [ ]:
change_metrics_query = f"""
INSERT INTO gold.daily_change_metrics (
    ref_date, table_name, total_inserts, total_updates,
    total_deletes, total_active_records
)
SELECT
    ref_date,
    table_name,
    countIf(operation_type = 'INSERT') AS total_inserts,
    countIf(operation_type = 'UPDATE') AS total_updates,
    countIf(operation_type = 'DELETE') AS total_deletes,
    (
        SELECT count()
        FROM silver.current_state AS cs
        WHERE cs.table_name = de.table_name
            AND cs.is_active = 1
    ) AS total_active_records
FROM silver.delta_events AS de
WHERE ref_date = toDate('{ref_date}')
GROUP BY ref_date, table_name
"""

client.execute_query(change_metrics_query)


In [ ]:
change_metrics_result = client.execute_query_with_result(
    f"""
    SELECT *
    FROM gold.daily_change_metrics
    WHERE ref_date = toDate('{ref_date}')
    """
)
change_metrics_result.result_rows


In [ ]:
quality_metrics_query = f"""
INSERT INTO gold.data_quality_metrics (
    ref_date, table_name, null_count, duplicate_count,
    total_records, quality_score
)
SELECT
    toDate('{ref_date}') AS ref_date,
    table_name,
    countIf(data = '' OR data IS NULL) AS null_count,
    count() - uniq(primary_key) AS duplicate_count,
    count() AS total_records,
    100.0 * (1 - (null_count + duplicate_count) / total_records) AS quality_score
FROM bronze.snapshot_raw
WHERE ref_date = toDate('{ref_date}')
GROUP BY table_name
"""

client.execute_query(quality_metrics_query)


In [ ]:
quality_metrics_result = client.execute_query_with_result(
    f"""
    SELECT *
    FROM gold.data_quality_metrics
    WHERE ref_date = toDate('{ref_date}')
    """
)
quality_metrics_result.result_rows


In [ ]:
historical_trends = client.execute_query_with_result(
    """
    SELECT
        ref_date,
        table_name,
        total_inserts,
        total_updates,
        total_deletes,
        total_active_records
    FROM gold.daily_change_metrics
    ORDER BY ref_date DESC, table_name
    LIMIT 50
    """
)
historical_trends.result_rows


In [ ]:
client.close()
